# Multiplexed state preparation by CORDIC arithmetic for the periodic Poisson isometry

The three listings of Section 6 of the paper, with the output printed there. They go from the circuit used as a library, through one wavenumber in its registers, to its cost as N grows.

Install the package first, from the repository root: `pip install -r requirements.txt`.


## Listing 1. The circuit at $N = 32$ and $F = 32$, simulated bit by bit at every wavenumber

The simplest use of the package: the configuration of the abstract. We build the circuit with `poisson_isometry`, simulate its arithmetic bit by bit for every wavenumber $k \ne 0$, and compare the prepared vector with $c(k)/\|c(k)\|_2$. The checks confirm that the work register returns to $|0\rangle$ and that the bits of $k$ are restored, which is what allows the circuit to be used inside a block encoding. The largest error is below the measured bound $6N\,2^{-F}$ of Section 4. Runs in a few seconds.


In [ ]:
import numpy as np
from quantumpoissonhomogenization.arithmetic import (
    poisson_isometry, exact_unit)

sv = poisson_isometry(n=5, F=32)       # N = 32, F = 32
amp, vreg, vex, _, _, ck = sv.simulate()   # every k
err = np.linalg.norm(amp - exact_unit(vex), axis=1)
r = sv.tree.rotation_counts()
tof = sum(sv.toffoli().values()) + r["toffoli"]

print(f"wavenumbers      = {len(err)}")
print(f"bits left set    = {ck['work_bits_left_set']}")
print(f"inputs restored  = {ck['inputs_restored']}")
print(f"max |error|      = {err.max():.1e}")
print(f"6 N 2^-F         = {6 * sv.N * 2.0**-sv.F:.1e}")
print(f"Toffoli gates    = {tof:,}")
print(f"ctrl. rotations  = {r['controlled_ry']}")


wavenumbers      = 1023
bits left set    = 0
inputs restored  = True
max |error|      = 9.0e-09
6 N 2^-F         = 4.5e-08
Toffoli gates    = 23,089
ctrl. rotations  = 65


## Listing 2. The example of Sections 2 and 3: one wavenumber, $k = (3, 1)$, at $N = 16$ and $F = 12$

One level down, to the registers. For one wavenumber, we print the integers $Z$ that Part 1 writes into the registers of $a$, $b$, and $\zeta$, each read as $Z/2^F$, and the vector that Part 2 prepares from them. The integers are the bit strings of the example in Section 2, and the prepared vector and its error are those of the example in Section 3. Runs in under a second.


In [ ]:
import numpy as np
from quantumpoissonhomogenization.arithmetic import (
    poisson_isometry, exact_unit)

sv = poisson_isometry(n=4, F=12)       # N = 16, F = 12
amp, vreg, vex, _, _, _ = sv.simulate()
k1, k2 = 3, 1
i = k1 * sv.N + k2 - 1       # k = 0 is skipped
Z = np.round(vreg[i, :3] * [1, 1, 3**0.5] * 2**sv.F)
exact = exact_unit(vex)[i]
d = np.linalg.norm(amp[i] - exact)

print(f"integers of a, b, zeta = {Z.astype(int)}")
print(f"prepared = {np.round(amp[i], 5)}")
print(f"exact    = {np.round(exact, 5)}")
print(f"error    = {d:.1e}")


integers of a, b, zeta = [2229  663  442]
prepared = [0.94709 0.28202 0.10835 0.10835]
exact    = [0.94703 0.28192 0.10876 0.10876]
error    = 5.9e-04


## Listing 3. The Toffoli count and work register of the circuit at $F = 45$ for $N = 16$ to $1024$

The property the circuit is built for: its cost does not depend on $N$. At $F = 45$, we build the circuit for $N = 16$ to $1024$ and print its Toffoli count and work register, with the number $3N^2$ of angles that a multiplexor or a table lookup must tabulate. The counts are those of Theorem 2 and Table 2 at every $N$, while the table grows to more than three million angles at $N = 1024$. At a fixed accuracy, the circuit grows with $N$ only through $F$, as in Table 3. Runs in about four seconds.


In [ ]:
from quantumpoissonhomogenization.arithmetic import (
    poisson_isometry)

for n in (4, 6, 8, 10):                # N = 16 to 1024
    sv = poisson_isometry(n, F=45)
    r = sv.tree.rotation_counts()
    tof = sum(sv.toffoli().values()) + r["toffoli"]
    work = sv.c.n - 2 * n + r["temporaries"]
    print(f"N = {2**n:4d}: Toffoli {tof:,},"
          f" work {work:,}, table {3 * 4**n:,}")


N =   16: Toffoli 44,424, work 5,672, table 768
N =   64: Toffoli 44,424, work 5,672, table 12,288
N =  256: Toffoli 44,424, work 5,672, table 196,608
N = 1024: Toffoli 44,424, work 5,672, table 3,145,728
